# Notebook 04-UNSW — SHAP Feature Attributions (Comprehensive)

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection  
**Author:** Md Anas Biswas, University of Portsmouth  
**Datasets covered here:** UNSW-NB15 only

## What this notebook does

1. Builds a **2000-sample stratified evaluation subsample** from the test set (matches NSL-KDD/CIC-IDS2017 sample size, gives ~18 U2R samples for SHAP stability on the rare class).
2. Computes SHAP values for all six canonical models on **uncalibrated probabilities**:
   - RF: `TreeExplainer` with `model_output='raw'`
   - XGBoost: native `pred_contribs` (bypasses the known SHAP library parsing bug)
   - DNN: `KernelExplainer` with 200-sample stratified background
3. Computes **per-class feature importance rankings** (Top-15) — separate rankings for Normal vs DoS vs Probe vs R2L vs U2R
4. **Robustness check (supplementary):** Re-runs SHAP for `xgb_5class_smote` on calibrated probabilities (500 samples) and compares top-15 rankings to the uncalibrated version. Documents whether calibration affects explanations.
5. Saves all SHAP arrays, indices, and rankings
6. Generates aggregate top-features figure and per-class figures
7. Commits and pushes

## Why SHAP on uncalibrated probabilities

SHAP explains the model's decision function, not the calibration-adjusted function. Explaining calibrated outputs would conflate model behaviour with post-hoc adjustment. This is the standard approach in the XAI literature (Krishna et al., 2022; Lundberg et al., 2017) and matches our NSL-KDD and CIC-IDS2017 analyses. The supplementary calibrated-SHAP check in §10 documents whether this choice meaningfully changes the explanations.

## Runtime budget (T4 GPU)

- RF × 2: ~10–15 min total (TreeExplainer is exact but slow)
- XGBoost × 2: ~1 min total (native pred_contribs)
- DNN × 2: ~30–40 min total (KernelExplainer is the bottleneck — model-agnostic, samples background)
- Calibrated robustness check: ~5 min
- **Total: 45–60 min**

Models save immediately after each one finishes so a disconnect mid-DNN doesn't lose tree results.

## 1. Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, shutil
shutil.copy('/content/drive/MyDrive/XIDS_Research/.gitconfig',       '/root/.gitconfig')
shutil.copy('/content/drive/MyDrive/XIDS_Research/.git-credentials', '/root/.git-credentials')

PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

!git config --global credential.helper 'store --file /content/drive/MyDrive/XIDS_Research/.git-credentials'
!git pull origin main

!pip install -q shap

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
!nvidia-smi -L

From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.

Device: cuda
GPU 0: Tesla T4 (UUID: GPU-bbf437bd-1f89-df02-6d2e-61623150b149)


## 2. Load data, models, calibration split indices

In [3]:
import numpy as np
import json
import joblib
from pathlib import Path

REPO       = Path(PROJECT_DIR)
PROC_DIR   = REPO / 'data/processed/unsw_nb15'
MODELS_DIR = REPO / 'models/unsw_nb15'
CALIB_DIR  = REPO / 'calibrators/unsw_nb15'
SHAP_DIR   = REPO / 'shap_values/unsw_nb15'
SHAP_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')
y_train_binary = np.load(PROC_DIR / 'y_train_binary.npy')
y_test_binary  = np.load(PROC_DIR / 'y_test_binary.npy')
y_train_5class = np.load(PROC_DIR / 'y_train_5class.npy')
y_test_5class  = np.load(PROC_DIR / 'y_test_5class.npy')

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)

# Calibration split — only use idx_eval for SHAP so stability tests in Notebook 05 are clean
idx_eval = np.load(CALIB_DIR / 'idx_eval.npy')

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_FEATURES = X_train.shape[1]

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'idx_eval (calibration eval half): {len(idx_eval):,} rows')
print(f'Total features (post one-hot): {N_FEATURES}')

X_train: (135341, 196)
X_test:  (63461, 196)
idx_eval (calibration eval half): 31,731 rows
Total features (post one-hot): 196


## 3. Build the 2000-sample stratified SHAP subsample

Drawn from `idx_eval` only (the calibration eval half), stratified by 5-class label so all classes — including U2R — are represented. The same 2000 indices are used across all six models so the explanations are directly comparable.

In [4]:
from sklearn.model_selection import train_test_split
from collections import Counter

SHAP_N = 2000

y_eval_5class = y_test_5class[idx_eval]

# Stratified subsample of size 2000 from idx_eval
shap_local_idx, _ = train_test_split(
    np.arange(len(idx_eval)),
    train_size=SHAP_N,
    stratify=y_eval_5class,
    random_state=42,
)
shap_local_idx = np.sort(shap_local_idx)
shap_idx = idx_eval[shap_local_idx]  # indices back into the full test set

X_shap = X_test[shap_idx]
y_shap_binary = y_test_binary[shap_idx]
y_shap_5class = y_test_5class[shap_idx]

np.save(SHAP_DIR / 'shap_subsample_idx.npy', shap_idx)

print(f'SHAP subsample shape: {X_shap.shape}')
print('\nPer-class breakdown of SHAP subsample:')
for cid in range(5):
    n = int((y_shap_5class == cid).sum())
    print(f'  {cid} {FIVE_CLASS_NAMES[cid]:8s}: {n:>4d}')

SHAP subsample shape: (2000, 196)

Per-class breakdown of SHAP subsample:
  0 Normal  : 1166
  1 DoS     :  129
  2 Probe   :  132
  3 R2L     :  560
  4 U2R     :   13


## 4. SHAP helper functions

In [5]:
import shap
import time
import xgboost as xgb

def shap_rf(model, X):
    """Random Forest via TreeExplainer (exact, slow on large samples)."""
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    # sklearn RF returns list of arrays (one per class) for multiclass
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)  # (n_samples, n_features, n_classes)
    elif sv.ndim == 3:
        # Some versions already return 3-D; ensure (n_samples, n_features, n_classes)
        if sv.shape[0] != X.shape[0]:
            sv = np.transpose(sv, (1, 2, 0))
    else:
        # Binary classifier in older versions can give (n_samples, n_features)
        sv = sv[..., np.newaxis]
    return sv

def shap_xgb(model, X, n_classes):
    """Native pred_contribs path — bypasses known SHAP library parsing bug."""
    dmat = xgb.DMatrix(X)
    booster = model.get_booster()
    contribs = booster.predict(dmat, pred_contribs=True)
    # contribs has shape (n_samples, n_features+1)            for binary
    #                 or (n_samples, n_classes, n_features+1) for multiclass
    if n_classes == 2:
        sv = contribs[:, :-1]  # drop bias term
        # Cast to (n_samples, n_features, n_classes=2) by computing the opposite sign
        # for class 0 to mirror the XGBoost convention.
        sv_full = np.stack([-sv, sv], axis=-1)
        return sv_full
    else:
        contribs = contribs[:, :, :-1]  # drop bias term, shape (N, n_classes, n_features)
        sv = np.transpose(contribs, (0, 2, 1))  # → (N, n_features, n_classes)
        return sv

def shap_dnn(state_dict, X_shap, X_bg, n_features, n_classes, batch_size=64):
    """DNN via KernelExplainer. Background = 200 stratified train samples.
    KernelExplainer is model-agnostic — we wrap forward() to return probabilities."""
    import torch.nn as nn
    import torch.nn.functional as F

    class XIDSMLP(nn.Module):
        def __init__(self, n_features, n_classes, p_drop=0.3):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p_drop),
                nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
                nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p_drop),
                nn.Linear(64, 32),          nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(p_drop),
                nn.Linear(32, n_classes),
            )
        def forward(self, x):
            return self.net(x)

    model = XIDSMLP(n_features, n_classes).to(device)
    model.load_state_dict(state_dict)
    model.eval()

    def predict_proba(arr):
        with torch.no_grad():
            t = torch.tensor(arr, dtype=torch.float32, device=device)
            out = F.softmax(model(t), dim=1)
        return out.cpu().numpy()

    explainer = shap.KernelExplainer(predict_proba, X_bg)
    # nsamples auto -> 2*F + 2048 — sufficient for 196 features
    sv = explainer.shap_values(X_shap, nsamples='auto', silent=True)
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)  # (n_samples, n_features, n_classes)
    return sv

print('SHAP helpers ready.')

SHAP helpers ready.


## 5. Build DNN background sets (200 stratified train samples per target type)

In [6]:
BG_N = 200

# Background for binary DNN — stratified by binary label
bg_binary_idx, _ = train_test_split(
    np.arange(len(y_train_binary)),
    train_size=BG_N, stratify=y_train_binary, random_state=42,
)
X_bg_binary = X_train[bg_binary_idx]

# Background for 5-class DNN — stratified by 5-class label
bg_5class_idx, _ = train_test_split(
    np.arange(len(y_train_5class)),
    train_size=BG_N, stratify=y_train_5class, random_state=42,
)
X_bg_5class = X_train[bg_5class_idx]

np.save(SHAP_DIR / 'bg_binary_idx.npy', bg_binary_idx)
np.save(SHAP_DIR / 'bg_5class_idx.npy', bg_5class_idx)

print(f'Binary DNN background: {X_bg_binary.shape}')
print(f'5-class DNN background: {X_bg_5class.shape}')

Binary DNN background: (200, 196)
5-class DNN background: (200, 196)


## 6. Compute SHAP for the four tree models (RF and XGBoost × binary/5-class)

In [7]:
# RF binary
name = 'rf_binary_cw'
print(f'{name}...')
t0 = time.time()
model = joblib.load(MODELS_DIR / f'{name}.joblib')
sv = shap_rf(model, X_shap)
np.save(SHAP_DIR / f'{name}_shap.npy', sv)
print(f'  Shape: {sv.shape}  Time: {time.time()-t0:.1f}s')

rf_binary_cw...
  Shape: (2000, 196, 2)  Time: 5232.5s


In [8]:
# XGB binary
name = 'xgb_binary_cw'
print(f'{name}...')
t0 = time.time()
model = joblib.load(MODELS_DIR / f'{name}.joblib')
sv = shap_xgb(model, X_shap, n_classes=2)
np.save(SHAP_DIR / f'{name}_shap.npy', sv)
print(f'  Shape: {sv.shape}  Time: {time.time()-t0:.1f}s')

xgb_binary_cw...
  Shape: (2000, 196, 2)  Time: 0.7s


In [ ]:
# RF 5-class
name = 'rf_5class_smote'
print(f'{name}...')
t0 = time.time()
model = joblib.load(MODELS_DIR / f'{name}.joblib')
sv = shap_rf(model, X_shap)
np.save(SHAP_DIR / f'{name}_shap.npy', sv)
print(f'  Shape: {sv.shape}  Time: {time.time()-t0:.1f}s')

rf_5class_smote...
  Shape: (2000, 196, 5)  Time: 13024.0s


In [ ]:
# XGB 5-class
name = 'xgb_5class_smote'
print(f'{name}...')
t0 = time.time()
model = joblib.load(MODELS_DIR / f'{name}.joblib')
sv = shap_xgb(model, X_shap, n_classes=5)
np.save(SHAP_DIR / f'{name}_shap.npy', sv)
print(f'  Shape: {sv.shape}  Time: {time.time()-t0:.1f}s')

xgb_5class_smote...
  Shape: (2000, 196, 5)  Time: 1.5s


## 7. Compute SHAP for the two DNNs (KernelExplainer)

This is the longest stage. KernelExplainer is model-agnostic — it perturbs the input across the background distribution. With 196 features, `nsamples='auto'` = 2×196+2048 = 2440 perturbations per data point. Times 2000 points means ~5M model evaluations per DNN. Expect 15–20 min per DNN on T4.

In [ ]:
# DNN binary
name = 'dnn_binary_cw'
print(f'{name}... (this will take 15–20 min)')
t0 = time.time()
state_dict = torch.load(MODELS_DIR / f'{name}.pt', map_location=device)
sv = shap_dnn(state_dict, X_shap, X_bg_binary, N_FEATURES, n_classes=2)
np.save(SHAP_DIR / f'{name}_shap.npy', sv)
print(f'  Shape: {sv.shape}  Time: {(time.time()-t0)/60:.1f} min')

dnn_binary_cw... (this will take 15–20 min)


  Shape: (2000, 196, 2)  Time: 12.6 min


In [ ]:
# DNN 5-class
name = 'dnn_5class_smote'
print(f'{name}... (this will take 15–20 min)')
t0 = time.time()
state_dict = torch.load(MODELS_DIR / f'{name}.pt', map_location=device)
sv = shap_dnn(state_dict, X_shap, X_bg_5class, N_FEATURES, n_classes=5)
np.save(SHAP_DIR / f'{name}_shap.npy', sv)
print(f'  Shape: {sv.shape}  Time: {(time.time()-t0)/60:.1f} min')

dnn_5class_smote... (this will take 15–20 min)


  Shape: (2000, 196, 5)  Time: 13.5 min


## 8. Aggregate feature importance rankings (Top-15 per model)

Importance = mean absolute SHAP value across all samples, averaged across classes for the predicted-class direction.

In [ ]:
MODEL_SPECS = {
    'rf_binary_cw':     {'target': 'binary',     'n_classes': 2},
    'xgb_binary_cw':    {'target': 'binary',     'n_classes': 2},
    'dnn_binary_cw':    {'target': 'binary',     'n_classes': 2},
    'rf_5class_smote':  {'target': 'multiclass', 'n_classes': 5},
    'xgb_5class_smote': {'target': 'multiclass', 'n_classes': 5},
    'dnn_5class_smote': {'target': 'multiclass', 'n_classes': 5},
}

TOP_K = 15
rankings_all = {}

for name, spec in MODEL_SPECS.items():
    sv = np.load(SHAP_DIR / f'{name}_shap.npy')
    # sv shape: (n_samples, n_features, n_classes)
    # Aggregate importance: mean of |SHAP| across samples, then mean across classes
    importance = np.abs(sv).mean(axis=0).mean(axis=-1)  # (n_features,)
    top_idx = np.argsort(importance)[::-1][:TOP_K]
    rankings_all[name] = [
        {'rank': i+1, 'feature': feature_names[idx], 'importance': float(importance[idx])}
        for i, idx in enumerate(top_idx)
    ]

with open(SHAP_DIR / 'feature_importance_rankings.json', 'w') as f:
    json.dump(rankings_all, f, indent=2)

print('Aggregate Top-15 feature rankings:\n')
for name, ranks in rankings_all.items():
    print(f'--- {name} ---')
    for r in ranks[:10]:
        print(f"  {r['rank']:>2d}. {r['feature']:35s}  {r['importance']:.4f}")
    print()

Aggregate Top-15 feature rankings:

--- rf_binary_cw ---
   1. sttl                                 0.0839
   2. ct_state_ttl                         0.0504
   3. dttl                                 0.0388
   4. ct_srv_dst                           0.0242
   5. state_INT                            0.0187
   6. ackdat                               0.0181
   7. ct_dst_src_ltm                       0.0177
   8. tcprtt                               0.0175
   9. dload                                0.0168
  10. smean                                0.0156

--- xgb_binary_cw ---
   1. sttl                                 3.4714
   2. ct_state_ttl                         0.8457
   3. sbytes                               0.5243
   4. smean                                0.4287
   5. ct_srv_dst                           0.3965
   6. dbytes                               0.3002
   7. ct_srv_src                           0.2581
   8. ct_dst_src_ltm                       0.2502
   9. ct_dst_sport_l

## 9. Per-class feature importance rankings

For each 5-class model, compute Top-15 features *separately* for each predicted class (Normal, DoS, Probe, R2L, U2R). This is the contribution that most IDS papers don't do at scale — per-class SHAP reveals which features distinguish each attack type, useful for SOC analyst dashboards.

In [ ]:
per_class_rankings = {}

for name in ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']:
    sv = np.load(SHAP_DIR / f'{name}_shap.npy')
    # sv shape: (n_samples, n_features, 5)
    per_class_rankings[name] = {}
    for cid, cname in enumerate(FIVE_CLASS_NAMES):
        # Only look at samples where the true class matches — gives the most diagnostic signal
        mask = (y_shap_5class == cid)
        if mask.sum() < 5:
            per_class_rankings[name][cname] = []  # skip if too few samples
            continue
        # Importance for this class: mean |SHAP[:, :, cid]| over the class-cid samples
        class_importance = np.abs(sv[mask, :, cid]).mean(axis=0)
        top_idx = np.argsort(class_importance)[::-1][:TOP_K]
        per_class_rankings[name][cname] = [
            {'rank': i+1, 'feature': feature_names[idx], 'importance': float(class_importance[idx])}
            for i, idx in enumerate(top_idx)
        ]

with open(SHAP_DIR / 'per_class_feature_rankings.json', 'w') as f:
    json.dump(per_class_rankings, f, indent=2)

# Print top-10 per class for XGBoost (the best 5-class performer)
print('Per-class Top-10 features for xgb_5class_smote:')
for cname, ranks in per_class_rankings['xgb_5class_smote'].items():
    print(f'\n  === {cname} ===')
    if not ranks:
        print('    (insufficient samples)')
        continue
    for r in ranks[:10]:
        print(f"    {r['rank']:>2d}. {r['feature']:35s}  {r['importance']:.4f}")

Per-class Top-10 features for xgb_5class_smote:

  === Normal ===
     1. sttl                                 3.7148
     2. ct_state_ttl                         0.5752
     3. sbytes                               0.3426
     4. smean                                0.3328
     5. ct_dst_src_ltm                       0.2632
     6. ct_srv_dst                           0.2608
     7. sloss                                0.1873
     8. synack                               0.1545
     9. sinpkt                               0.1456
    10. dmean                                0.1448

  === DoS ===
     1. ct_dst_sport_ltm                     0.3059
     2. smean                                0.2859
     3. ct_src_ltm                           0.1975
     4. ct_srv_dst                           0.1734
     5. sbytes                               0.1595
     6. proto_udp                            0.1290
     7. sloss                                0.1244
     8. sttl                       

## 10. Robustness check — calibrated SHAP for XGBoost 5-class (500 samples)

Defends against the reviewer question: *does calibration change the explanations?* We compute SHAP on calibrated probabilities for one model (XGBoost 5-class, the best performer) and check whether the top-15 feature ranking changes.

Isotonic regression is monotonic, so it preserves rank ordering of model outputs but can shift attribution magnitudes. Expected result: same top features, possibly slightly different ordering — confirming that explanations are robust to the calibration choice.

In [ ]:
# Smaller subsample (500) for the robustness check — drawn from the same SHAP subsample for direct comparison
ROBUST_N = 500
robust_local, _ = train_test_split(
    np.arange(SHAP_N), train_size=ROBUST_N,
    stratify=y_shap_5class, random_state=42,
)
robust_local = np.sort(robust_local)
X_robust = X_shap[robust_local]
y_robust_5class = y_shap_5class[robust_local]

# Load XGBoost model and its calibrator
xgb_model = joblib.load(MODELS_DIR / 'xgb_5class_smote.joblib')
calibrators = joblib.load(CALIB_DIR / 'xgb_5class_smote_calibrators.joblib')

def calibrated_predict_proba(arr):
    proba_raw = xgb_model.predict_proba(arr)
    out = np.zeros_like(proba_raw)
    for c, ir in enumerate(calibrators):
        out[:, c] = ir.predict(proba_raw[:, c])
    row_sums = out.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return out / row_sums

# KernelExplainer with 100 background samples drawn stratified from training
bg_robust_idx, _ = train_test_split(
    np.arange(len(y_train_5class)),
    train_size=100, stratify=y_train_5class, random_state=42,
)
X_bg_robust = X_train[bg_robust_idx]

print(f'Computing calibrated SHAP for xgb_5class_smote on {ROBUST_N} samples (this takes ~5 min)...')
t0 = time.time()
robust_explainer = shap.KernelExplainer(calibrated_predict_proba, X_bg_robust)
sv_calibrated = robust_explainer.shap_values(X_robust, nsamples='auto', silent=True)
if isinstance(sv_calibrated, list):
    sv_calibrated = np.stack(sv_calibrated, axis=-1)
print(f'  Done in {(time.time()-t0)/60:.1f} min  Shape: {sv_calibrated.shape}')

np.save(SHAP_DIR / 'xgb_5class_smote_shap_calibrated.npy', sv_calibrated)

In [ ]:
# Compare top-15 features uncalibrated vs calibrated
sv_uncal_full = np.load(SHAP_DIR / 'xgb_5class_smote_shap.npy')
sv_uncal = sv_uncal_full[robust_local]  # same 500 samples as the calibrated run

importance_uncal = np.abs(sv_uncal).mean(axis=0).mean(axis=-1)
importance_cal   = np.abs(sv_calibrated).mean(axis=0).mean(axis=-1)

top_uncal = np.argsort(importance_uncal)[::-1][:TOP_K]
top_cal   = np.argsort(importance_cal)[::-1][:TOP_K]

uncal_set = set(top_uncal)
cal_set   = set(top_cal)
jaccard = len(uncal_set & cal_set) / len(uncal_set | cal_set)

# Spearman rank correlation between uncalibrated and calibrated importance vectors
from scipy.stats import spearmanr
rho, _ = spearmanr(importance_uncal, importance_cal)

print(f'Top-{TOP_K} Jaccard similarity (uncal vs cal): {jaccard:.3f}')
print(f'Spearman rank correlation on full feature importance: {rho:.4f}')

print(f'\n{"Rank":<6}{"Uncalibrated":<35}{"Calibrated":<35}{"Match"}')
print('-' * 85)
for i in range(TOP_K):
    f_u = feature_names[top_uncal[i]]
    f_c = feature_names[top_cal[i]]
    match = '✓' if f_u == f_c else (' (in top-{})'.format(TOP_K) if top_uncal[i] in cal_set else '✗')
    print(f'{i+1:<6}{f_u:<35}{f_c:<35}{match}')

robust_summary = {
    'n_samples': ROBUST_N,
    'jaccard_top_k': float(jaccard),
    'spearman_rho': float(rho),
    'top_features_uncalibrated': [feature_names[i] for i in top_uncal],
    'top_features_calibrated':   [feature_names[i] for i in top_cal],
}
with open(SHAP_DIR / 'calibration_robustness_check.json', 'w') as f:
    json.dump(robust_summary, f, indent=2)

## 11. Save the comparison table + per-class CSV

In [ ]:
import pandas as pd

# Aggregate top-15 across all six models — wide table for the paper
rows = []
for rank in range(TOP_K):
    row = {'Rank': rank + 1}
    for name in MODEL_SPECS:
        row[name] = rankings_all[name][rank]['feature']
    rows.append(row)
top15_df = pd.DataFrame(rows)
agg_path = REPO / 'results/tables/unsw_top15_features_by_model.csv'
top15_df.to_csv(agg_path, index=False)
print('Top-15 features by model:')
print(top15_df.to_string(index=False))
print(f'\nSaved: {agg_path}')

# Per-class top-10 for XGBoost 5-class (the best 5-class performer)
rows = []
for rank in range(10):
    row = {'Rank': rank + 1}
    for cname in FIVE_CLASS_NAMES:
        ranks = per_class_rankings['xgb_5class_smote'].get(cname, [])
        row[cname] = ranks[rank]['feature'] if rank < len(ranks) else ''
    rows.append(row)
pc_df = pd.DataFrame(rows)
pc_path = REPO / 'results/tables/unsw_xgb5_per_class_top10.csv'
pc_df.to_csv(pc_path, index=False)
print('\nXGBoost 5-class per-class Top-10:')
print(pc_df.to_string(index=False))
print(f'\nSaved: {pc_path}')

## 12. Visualisations

In [ ]:
import matplotlib.pyplot as plt

# Figure 1: top-15 aggregate importance per model (6-panel)
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
model_order = [
    'rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
    'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
]

for ax, name in zip(axes.flat, model_order):
    ranks = rankings_all[name][:TOP_K]
    features = [r['feature'] for r in ranks][::-1]
    importances = [r['importance'] for r in ranks][::-1]
    color = '#2E86AB' if 'binary' in name else '#E63946'
    ax.barh(features, importances, color=color)
    ax.set_title(f'{name}  (Top-{TOP_K})', fontsize=10)
    ax.set_xlabel('Mean |SHAP|')
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('UNSW-NB15 — Aggregate Top-15 Feature Importance by Model', fontsize=14, y=1.00)
plt.tight_layout()
fig1_path = REPO / 'results/figures/unsw_shap_top15_per_model.png'
plt.savefig(fig1_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {fig1_path}')

In [ ]:
# Figure 2: per-class Top-10 for XGBoost 5-class (the strongest 5-class model)
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
for ax, cname in zip(axes.flat, FIVE_CLASS_NAMES):
    ranks = per_class_rankings['xgb_5class_smote'].get(cname, [])
    if not ranks:
        ax.text(0.5, 0.5, '(too few samples)', ha='center', va='center')
        ax.set_title(cname)
        continue
    features = [r['feature'] for r in ranks[:10]][::-1]
    importances = [r['importance'] for r in ranks[:10]][::-1]
    ax.barh(features, importances, color='#9B5DE5')
    ax.set_title(f'{cname}', fontsize=11)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlabel('|SHAP|')

plt.suptitle('UNSW-NB15 — XGBoost 5-class per-class Top-10 features', fontsize=13, y=1.02)
plt.tight_layout()
fig2_path = REPO / 'results/figures/unsw_shap_xgb5_per_class.png'
plt.savefig(fig2_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {fig2_path}')

In [ ]:
# Figure 3: calibrated vs uncalibrated SHAP comparison (robustness check)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: top-15 uncalibrated
feats_u = [feature_names[i] for i in top_uncal][::-1]
imps_u  = [importance_uncal[i] for i in top_uncal][::-1]
axes[0].barh(feats_u, imps_u, color='#E63946')
axes[0].set_title('Uncalibrated SHAP\nTop-15 features')
axes[0].set_xlabel('Mean |SHAP|')
axes[0].tick_params(axis='y', labelsize=8)
axes[0].grid(axis='x', alpha=0.3)

# Right: top-15 calibrated
feats_c = [feature_names[i] for i in top_cal][::-1]
imps_c  = [importance_cal[i] for i in top_cal][::-1]
axes[1].barh(feats_c, imps_c, color='#2E86AB')
axes[1].set_title(f'Calibrated SHAP\nTop-15 features (Jaccard={jaccard:.2f}, ρ={rho:.2f})')
axes[1].set_xlabel('Mean |SHAP|')
axes[1].tick_params(axis='y', labelsize=8)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle(f'xgb_5class_smote — Calibration robustness check (n={ROBUST_N})', fontsize=12)
plt.tight_layout()
fig3_path = REPO / 'results/figures/unsw_shap_calibration_robustness.png'
plt.savefig(fig3_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {fig3_path}')

## 13. Commit and push

In [ ]:
os.chdir(PROJECT_DIR)
!git add notebooks/04_unsw_shap.ipynb \
         results/figures/unsw_shap_top15_per_model.png \
         results/figures/unsw_shap_xgb5_per_class.png \
         results/figures/unsw_shap_calibration_robustness.png \
         results/tables/unsw_top15_features_by_model.csv \
         results/tables/unsw_xgb5_per_class_top10.csv
!git commit -m "Notebook 04-UNSW: SHAP analysis (2000-sample, uncalibrated + per-class rankings + calibration robustness check)"
!git push origin main

---

## What to check

1. **All six SHAP arrays saved** to `shap_values/unsw_nb15/`.
2. **Jaccard top-15 (uncalibrated vs calibrated) ≥ 0.7** — expected. If <0.5, calibration is dramatically changing explanations and the paper needs a caveat.
3. **Spearman ρ on full importance ≥ 0.9** — confirms isotonic preserves global feature ranking.
4. **Per-class top-10 features differ across classes** — confirms per-class SHAP is meaningful (vs all classes producing the same ranking, which would mean the per-class breakdown is uninformative).

## What to expect in the top-15 features

On UNSW-NB15, the literature consistently surfaces these as top features:
- `dur` (connection duration)
- `sbytes`, `dbytes` (source/destination bytes)
- `sttl`, `dttl` (TTL values)
- `smean`, `dmean` (mean packet sizes)
- `rate`, `sload`, `dload` (rate-based features)
- `ct_srv_src`, `ct_srv_dst`, `ct_dst_ltm` (connection-count features)

Plus one-hot encoded `proto_*` and `service_*` features for connection-type discrimination.

## Next step

**Notebook 05-UNSW** — explanation stability under perturbation (Gaussian / FGSM / PGD).